# Étape 2 — Modélisation thématique (LDA & BERTopic)

**Objectif :** Identifier les structures thématiques latentes du corpus,  
mesurer la spécialisation partisane par thème et préparer les distributions  
de topics pour l'analyse de cadrage (étape 3).

Deux approches complémentaires :
- **LDA** (Latent Dirichlet Allocation) — modèle probabiliste bag-of-words
- **BERTopic** — clustering d'embeddings multilingues (SentenceTransformer)

## 0. Imports

In [ ]:
import json
import sys
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from wordcloud import WordCloud

# Gensim / LDA
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel

# BERTopic
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer

import spacy

sys.path.append(str(Path("../src")))
from function import build_lemmatizer

warnings.filterwarnings("ignore")

DATA_DIR    = Path("../data")
PROC_DIR    = DATA_DIR / "processed"
FIGURES_DIR = Path("../figures")
FIGURES_DIR.mkdir(exist_ok=True)

CORPUS_CSV   = PROC_DIR / "corpus_1993.csv"
TOKENS_JSONL = PROC_DIR / "tokens_1993.jsonl"

print("Corpus CSV  :", CORPUS_CSV.exists())
print("Tokens JSONL:", TOKENS_JSONL.exists())

## 1. Chargement des données

In [ ]:
df = pd.read_csv(CORPUS_CSV)
print(f"Documents chargés : {len(df):,}")
print(f"Familles partisanes : {df['titulaire-soutien-simplifie'].nunique()}")

# Chargement des tokens lemmatisés
tokens_map = {}
with open(TOKENS_JSONL, encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        tokens_map[row["id"]] = row["tokens"]

df["tokens"] = df["id"].map(tokens_map)
df = df[df["tokens"].notna()].reset_index(drop=True)

print(f"Documents avec tokens : {len(df):,}")

## 2. LDA — Latent Dirichlet Allocation

### 2.1 Construction du dictionnaire et du corpus Gensim

On filtre les termes très rares (< 10 documents) et trop fréquents (> 60 % du corpus)  
afin de ne conserver que les mots porteurs d'information thématique.

In [ ]:
tokenized_docs = df["tokens"].tolist()

dictionary = corpora.Dictionary(tokenized_docs)
print(f"Vocabulaire avant filtrage : {len(dictionary):,}")

dictionary.filter_extremes(no_below=10, no_above=0.60)
print(f"Vocabulaire après filtrage : {len(dictionary):,}")

corpus_bow = [dictionary.doc2bow(tokens) for tokens in tokenized_docs]

### 2.2 Sélection du nombre de topics — cohérence C_v

On entraîne des modèles LDA pour K ∈ {5, 8, 10, 12, 15} et on compare  
le score de cohérence C_v. On choisit le K maximisant la cohérence  
tout en garantissant l'interprétabilité.

In [ ]:
K_VALUES  = [5, 8, 10, 12, 15]
coherences = []

for k in K_VALUES:
    lda = LdaModel(
        corpus=corpus_bow, id2word=dictionary,
        num_topics=k, passes=10, random_state=42,
        alpha="auto", eta="auto"
    )
    cm = CoherenceModel(
        model=lda, texts=tokenized_docs,
        dictionary=dictionary, coherence="c_v"
    )
    coherences.append(cm.get_coherence())
    print(f"  K={k:2d}  C_v = {coherences[-1]:.4f}")

best_k = K_VALUES[int(np.argmax(coherences))]
print(f"\n→ Meilleur K : {best_k}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(K_VALUES, coherences, "o-", color="steelblue", lw=2)
ax.axvline(best_k, color="red", linestyle="--", label=f"K optimal = {best_k}")
ax.set_xlabel("Nombre de topics"); ax.set_ylabel("Cohérence C_v")
ax.set_title("Sélection du nombre de topics — LDA")
ax.legend(); plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_lda_coherence.png", dpi=150, bbox_inches="tight")
plt.show()

### 2.3 Entraînement du modèle LDA final

In [ ]:
lda_model = LdaModel(
    corpus=corpus_bow, id2word=dictionary,
    num_topics=best_k, passes=20, random_state=42,
    alpha="auto", eta="auto"
)

print(f"Modèle LDA entraîné ({best_k} topics, 20 passes)\n")
for i, topic in lda_model.print_topics(num_words=12):
    print(f"Topic {i:2d} : {topic}\n")

### 2.4 Nommage et interprétation des topics

Après examen des termes représentatifs, on attribue un label thématique  
à chaque topic. **Adapter les labels ci-dessous après inspection.**

In [ ]:
# ── À ADAPTER selon les topics obtenus ──────────────────────────────────────
# Exemple pour K=10 :
TOPIC_LABELS = {
    0: "Économie & emploi",
    1: "Environnement & écologie",
    2: "Sécurité & immigration",
    3: "Services publics & local",
    4: "Europe & international",
    5: "Gauche & travail",
    6: "Droite & valeurs",
    7: "Jeunesse & éducation",
    8: "Agriculture & ruralité",
    9: "Démocratie & institutions",
}
# Tronquer ou compléter si best_k != 10
TOPIC_LABELS = {k: v for k, v in TOPIC_LABELS.items() if k < best_k}

# Affichage des top mots par topic avec label
for topic_id, label in TOPIC_LABELS.items():
    words = [w for w, _ in lda_model.show_topic(topic_id, topn=8)]
    print(f"[{topic_id:2d}] {label:<35} : {', '.join(words)}")

### 2.5 Distribution des topics par document

In [ ]:
# Vecteur de distribution de topics pour chaque document
def get_topic_vector(bow, model, n_topics):
    dist = dict(model.get_document_topics(bow, minimum_probability=0.0))
    return [dist.get(i, 0.0) for i in range(n_topics)]

topic_vectors = np.array([
    get_topic_vector(bow, lda_model, best_k) for bow in corpus_bow
])

# Ajouter au DataFrame
for i in range(best_k):
    label = TOPIC_LABELS.get(i, f"Topic_{i}")
    df[f"lda_topic_{i}"] = topic_vectors[:, i]

df["lda_dominant_topic"] = topic_vectors.argmax(axis=1)
df["lda_dominant_label"] = df["lda_dominant_topic"].map(TOPIC_LABELS)

print("Distribution du topic dominant :")
print(df["lda_dominant_label"].value_counts().to_string())

### 2.6 Spécialisation thématique par parti

In [ ]:
topic_cols = [f"lda_topic_{i}" for i in range(best_k)]
labels     = [TOPIC_LABELS.get(i, f"Topic_{i}") for i in range(best_k)]

# Distribution moyenne des topics par parti
parti_topic = (
    df.groupby("titulaire-soutien-simplifie")[topic_cols]
    .mean()
    .rename(columns={f"lda_topic_{i}": TOPIC_LABELS.get(i, f"Topic_{i}") for i in range(best_k)})
)

# Indice de spécialisation : écart à la distribution globale
global_dist = df[topic_cols].mean().values
spec_matrix = parti_topic.values / (global_dist + 1e-9)  # ratio
df_spec = pd.DataFrame(spec_matrix, index=parti_topic.index, columns=labels)

# Heatmap distribution
fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(
    parti_topic, cmap="YlOrRd", ax=ax, linewidths=0.4,
    cbar_kws={"label": "Proportion moyenne"}, fmt=".3f", annot=True
)
ax.set_title("Distribution moyenne des topics LDA par parti")
ax.set_xlabel("Topic"); ax.set_ylabel("Parti")
plt.xticks(rotation=45, ha="right"); plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_lda_topic_by_parti.png", dpi=150, bbox_inches="tight")
plt.show()

# Heatmap spécialisation
fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(
    df_spec, cmap="RdBu_r", center=1.0, ax=ax, linewidths=0.4,
    cbar_kws={"label": "Indice de spécialisation (ratio / moyenne globale)"},
    fmt=".2f", annot=True
)
ax.set_title("Indice de spécialisation thématique par parti (LDA)")
ax.set_xlabel("Topic"); ax.set_ylabel("Parti")
plt.xticks(rotation=45, ha="right"); plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_lda_specialisation.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. BERTopic

### 3.1 Calcul des embeddings

On utilise le modèle multilingue `paraphrase-multilingual-MiniLM-L12-v2`  
(384 dimensions, entraîné sur 50+ langues dont le français).

> ⏱ Le calcul des embeddings prend ~15 s sur GPU (ou ~3 min sur CPU).

In [ ]:
EMB_FILE = PROC_DIR / "embeddings_1993.npy"

docs_emb = df["clean_text_embeddings"].tolist()

if EMB_FILE.exists():
    embeddings = np.load(EMB_FILE)
    print(f"Embeddings chargés depuis le cache : {embeddings.shape}")
else:
    device = "cuda"  # remplacer par "cpu" si pas de GPU
    embedding_model = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        device=device
    )
    embeddings = embedding_model.encode(
        docs_emb, batch_size=64, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=True
    )
    np.save(EMB_FILE, embeddings)
    print(f"Embeddings calculés et sauvegardés : {embeddings.shape}")

### 3.2 Pipeline BERTopic

In [ ]:
nlp        = spacy.load("fr_core_news_md")
stop_words = list(nlp.Defaults.stop_words) + [
    "france", "français", "francaise", "francais",
    "circonscription", "elections", "election", "legislatives",
    "mars", "ans", "candidat", "candidats", "suppléant", "suppléante",
    "votez", "voter", "vu", "madame", "monsieur", "mademoiselle",
    "notre", "nos", "votre", "vos", "politique", "pays",
    "po", "fonds", "cevipof", "cevipov",
]

umap_model = UMAP(
    n_neighbors=15, n_components=5, min_dist=0.0,
    metric="cosine", random_state=42
)
hdbscan_model = HDBSCAN(
    min_cluster_size=80, min_samples=10,
    metric="euclidean", cluster_selection_method="eom",
    prediction_data=True
)
vectorizer_model = CountVectorizer(
    stop_words=stop_words, ngram_range=(1, 2),
    min_df=3, max_df=0.70
)

topic_model = BERTopic(
    embedding_model=None,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    min_topic_size=80,
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs_emb, embeddings)

topic_info = topic_model.get_topic_info()
print(f"\nNombre de topics détectés : {len(topic_info) - 1}  (hors bruit -1)")
print(f"Documents non classés (-1) : {(np.array(topics) == -1).sum()}")
topic_info.head(15)

### 3.3 Réduction et nettoyage des topics

In [ ]:
# Réduction à un nombre interprétable de topics
N_TOPICS_FINAL = 12

topic_model.reduce_topics(docs_emb, nr_topics=N_TOPICS_FINAL)
topics_reduced = topic_model.topics_

topic_info_r = topic_model.get_topic_info()
print(f"Topics après réduction : {len(topic_info_r) - 1}")
topic_info_r.head(15)

### 3.4 Visualisations BERTopic

In [ ]:
# Barchart des top mots par topic
fig_bar = topic_model.visualize_barchart(top_n_topics=N_TOPICS_FINAL, n_words=8, height=700)
fig_bar.write_html(str(FIGURES_DIR / "02_bertopic_barchart.html"))
fig_bar.show()

# Carte inter-topics
fig_map = topic_model.visualize_topics()
fig_map.write_html(str(FIGURES_DIR / "02_bertopic_intertopic_map.html"))
fig_map.show()

### 3.5 Distribution des topics BERTopic par parti

In [ ]:
df["bertopic"] = topics_reduced

# Exclure bruit (-1)
df_bt = df[df["bertopic"] != -1].copy()

# Noms courts des topics
bt_names = {
    row["Topic"]: " / ".join(row["Representation"][:3])
    for _, row in topic_model.get_topic_info().iterrows()
    if row["Topic"] != -1
}
df_bt["bertopic_label"] = df_bt["bertopic"].map(bt_names)

# Distribution par parti
bt_dist = (
    df_bt.groupby(["titulaire-soutien-simplifie", "bertopic_label"])
    .size().unstack(fill_value=0)
)
bt_dist_pct = bt_dist.div(bt_dist.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(15, 7))
sns.heatmap(
    bt_dist_pct, cmap="Blues", ax=ax, linewidths=0.3,
    cbar_kws={"label": "Proportion"},
    fmt=".2f", annot=True
)
ax.set_title("Distribution des topics BERTopic par parti")
plt.xticks(rotation=45, ha="right"); plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_bertopic_by_parti.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Synthèse comparative LDA / BERTopic

| Critère | LDA | BERTopic |
|---------|-----|----------|
| Type de représentation | Bag-of-words (TF-IDF) | Embeddings sémantiques |
| Topics | Distributions de mots | Clusters de documents |
| Interprétabilité | Bonne (mots) | Bonne (mots + documents) |
| Bruit | Aucun | Documents non classés (-1) |
| Sensibilité à l'OCR | Modérée | Faible (embeddings robustes) |

In [ ]:
# Résumé chiffré
print("─" * 55)
print(f"LDA  — {best_k} topics, cohérence C_v : {max(coherences):.4f}")
print(f"BERT — {len(topic_info_r)-1} topics, {(np.array(topics_reduced)==-1).sum()} docs non classés")
print("─" * 55)

## 5. Sauvegarde des résultats

In [ ]:
# Ajouter les colonnes BERTopic et sauvegarder
df["bertopic"]       = topics_reduced
df["bertopic_label"] = df["bertopic"].map(bt_names).fillna("bruit")

COLS_OUT = (
    ["id", "titulaire-soutien-simplifie", "titulaire-sexe",
     "lda_dominant_topic", "lda_dominant_label", "bertopic", "bertopic_label"]
    + [f"lda_topic_{i}" for i in range(best_k)]
)

df[COLS_OUT].to_csv(PROC_DIR / "topics_1993.csv", index=False)
print("✓ Sauvegardé :", PROC_DIR / "topics_1993.csv")

# Sauvegarder le modèle BERTopic
topic_model.save(str(PROC_DIR / "bertopic_model_1993"))
print("✓ Modèle BERTopic sauvegardé :", PROC_DIR / "bertopic_model_1993")